In [2]:
import pandas as pd


import re
import ast

from pathlib import Path

import ollama


# Cargar dataset que generamos con FinBert


A fin de que los resultados fueran comparables, definimos para esta prueba trabajar con las mismas noticias de 512 tokens o menos que utilizamos para la obtención del sentimiento financiero con Finbert.

Aprovechamos esta carga para limpiar el dataset de algunas columnas que no iban a ser necesarias y decidimos guardar los resultados de esta prueba con ollama, junto con valores de los sentimientos financieros que habíamos obtenido con Finbert.

In [3]:
df_news = pd.read_csv('../outputs/df_news_1_chunk_con_sentimiento_v1_incremental.csv')


In [7]:
df_news.drop(columns=['pos_chunk_2', 'neu_chunk_2', 'neg_chunk_2', 'pos_chunk_3', 'neu_chunk_3', 'neg_chunk_3'], inplace=True)

In [8]:
df_news['len_tickers'] = df_news['tickers_encontrados'].apply(len)
df_news['tickers_encontrados'][df_news['len_tickers'] > 1]

0                      ['UBER']
1              ['TSLA', 'UBER']
2       ['GOOG', 'NVDA', 'SPY']
3                        ['GS']
4       ['GOOG', 'NVDA', 'SPY']
                 ...           
5913                   ['TSLA']
5914                    ['BRK']
5915                   ['TSLA']
5916            ['AMD', 'NVDA']
5917                   ['AAPL']
Name: tickers_encontrados, Length: 5918, dtype: object

# Selección del Modelo

Lo que buscamos con esta prueba fue obtener un nuevo regresor para nuestras predicciones pidiéndole vía prompt a un LLM que nos hiciera una recomendación de qué tanto nos convenía invertir en una acción según lo que leía sobre esa acción en la noticia.
Le pedimos que nos diera un puntaje de 1 a 5 donde:

1 = Strongly Negative (Do not invest)
3 = Neutral
5 = Strongly Positive (Strong Buy)


Para esto, buscamos dentro de ollama un modelo entrenado con datos financieros. Si bien no teníamos confianza en que estas recomendaciones pudieran servir para mejorar las predicciones, de todos modos consideramos que utilizar un modelo de este tipo era lo más apropiado.

[finance-llama-8b:fp16](https://ollama.com/martain7r/finance-llama-8b:fp16)

[Model Card for Finance-Llama-8B (Hugging Face)](https://huggingface.co/tarun7r/Finance-Llama-8B)

# Test funcionamiento Ollama

In [9]:
ticker_ejemplo = df_news.iloc[0, df_news.columns.get_loc('tickers_encontrados')]
texto_ejemplo = df_news.iloc[0, df_news.columns.get_loc('texto_full')]
texto_ejemplo

'Technology Sector Update for 08/01/2023: ZBRA, ANET, UBER, XLK, XSD https://www.nasdaq.com/articles/technology-sector-update-for-08-01-2023%3A-zbra-anet-uber-xlk-xsd Technology stocks were slipping premarket Tuesday, with the Technology Select Sector SPDR Fund (XLK) 0.5% lower and the SPDR S&P Semiconductor ETF (XSD) declining by 0.9% recently.\nZebra Technologies (ZBRA) was shedding over 18% in value after it reported fiscal Q2 non-GAAP earnings of $3.29 per diluted share, down from $4.61 a year earlier. Analysts polled by Capital IQ expected $3.28.\nArista Networks (ANET) was rallying by more than 14% after it reported Q2 adjusted earnings of $1.58 per diluted share, up from $1.08 a year earlier. Analysts polled by Capital IQ expected $1.44.\nUber Technologies (UBER) was up more than 1% after it reported a Q2 net income of $0.18 per diluted share, swinging from a loss of $1.33 a year earlier. Analysts polled by Capital IQ expected a per-share loss of $0.01.\nThe views and opinions e

'Technology Sector Update for 08/01/2023: ZBRA, ANET, UBER, XLK, XSD https://www.nasdaq.com/articles/technology-sector-update-for-08-01-2023%3A-zbra-anet-uber-xlk-xsd Technology stocks were slipping premarket Tuesday, with the Technology Select Sector SPDR Fund (XLK) 0.5% lower and the SPDR S&P Semiconductor ETF (XSD) declining by 0.9% recently.\nZebra Technologies (ZBRA) was shedding over 18% in value after it reported fiscal Q2 non-GAAP earnings of $3.29 per diluted share, down from $4.61 a year earlier. Analysts polled by Capital IQ expected $3.28.\nArista Networks (ANET) was rallying by more than 14% after it reported Q2 adjusted earnings of $1.58 per diluted share, up from $1.08 a year earlier. Analysts polled by Capital IQ expected $1.44.\nUber Technologies (UBER) was up more than 1% after it reported a Q2 net income of $0.18 per diluted share, swinging from a loss of $1.33 a year earlier. Analysts polled by Capital IQ expected a per-share loss of $0.01.\nThe views and opinions expressed herein are the views and opinions of the author and do not necessarily reflect those of Nasdaq, Inc.'

In [10]:
ticker_ejemplo

"['UBER']"

In [11]:
prompt = f"""
For academic research, rate the article’s tone regarding investment in the company {ticker_ejemplo} 
on a scale from 1 to 5, where:
1 = Strongly Negative (Do not invest)
3 = Neutral
5 = Strongly Positive (Strong Buy)
Respond only with the number.
Text:
{texto_ejemplo}
"""

In [ ]:
response = ollama.chat(model="martain7r/finance-llama-8b:fp16", messages=[{"role": "user", "content": prompt}])
print(response)

model='martain7r/finance-llama-8b:fp16' created_at='2025-11-16T01:41:11.2096104Z' done=True done_reason='stop' total_duration=37214902600 load_duration=18722624200 prompt_eval_count=491 prompt_eval_duration=17589650500 eval_count=3 eval_duration=892582100 message=Message(role='assistant', content=' 3', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None


In [13]:
response["message"]["content"]

' 3'

In [14]:
def extraer_entero_1_a_5(texto):
    """
    Extrae el primer número del string, lo convierte a entero
    y valida que esté entre 1 y 5.
    Devuelve el entero o None si no hay número válido.
    """
    match = re.search(r"[-+]?\d*\.?\d+", texto)
    if not match:
        return None
    
    # Convertimos a float primero, luego a int
    numero = int(float(match.group()))
    
    # Validamos el rango 1–5
    if 1 <= numero <= 5:
        return numero
    
    return None

In [15]:
puntaje_ejemplo = extraer_entero_1_a_5(response["message"]["content"])

puntaje_ejemplo


3

# Pipeline para obtener recomendación de inversión con Ollama (martain7r/finance-llama-8b:fp16)

In [ ]:
#Convertir string-list a lista
def parse_tickers(x):
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except:
        return []

#Extraer número entre 1 y 5
def extraer_entero_1_a_5(texto):
    match = re.search(r"[-+]?\d*\.?\d+", texto)
    if not match:
        return None
    numero = int(float(match.group()))
    return numero if 1 <= numero <= 5 else None

#Función principal
def procesar_news(
    df,
    salida_ok="resultados_recomendaciones.csv",
    salida_err="errores_recomendaciones.csv",
    modelo="martain7r/finance-llama-8b:fp16"
):
    """
    Recorre un dataframe con columnas:
    - 'tickers_encontrados' (str o list)
    - 'texto_full'

    Genera un nuevo DF con un registro por ticker y columna recomendacion_inversion.
    Guarda resultados y errores en CSVs incrementales.
    Incluye en el csv la información del df de entrada
    """

    # Crear archivos si no existen
    Path(salida_ok).touch(exist_ok=True)
    Path(salida_err).touch(exist_ok=True)

    resultados = []
    errores = []

    for idx, row in df.iterrows():
        try:
            lista_tickers = parse_tickers(row["tickers_encontrados"])
            texto = row["texto_full"]

            for ticker in lista_tickers:

                prompt = f"""
                    For academic research, rate the article’s tone regarding investment in the company {ticker}
                    on a scale from 1 to 5, where:
                    1 = Strongly Negative (Do not invest)
                    3 = Neutral
                    5 = Strongly Positive (Strong Buy)
                    Respond only with the number.
                    Text:
                    {texto}
                """

                # LLM 
                resp = ollama.chat(
                    model=modelo,
                    messages=[{"role": "user", "content": prompt}]
                )
                raw_output = resp["message"]["content"]

                # Parseo del output
                score = extraer_entero_1_a_5(raw_output)

                if score is None:
                    errores.append({
                        "idx_original": idx,
                        "ticker": ticker,
                        "raw_output": raw_output,
                        "reason": "no se pudo parsear score"
                    })
                    continue

                # Guardar resultado correcto
                fila = row.to_dict()
                fila["ticker_individual"] = ticker
                fila["recomendacion_inversion"] = score
                resultados.append(fila)

                #  Guardado incremental (crea el header sólo si el archivo está vacío)
                file_ok = Path(salida_ok)
                write_header = not file_ok.exists() or file_ok.stat().st_size == 0

                pd.DataFrame([fila]).to_csv(
                    salida_ok,
                    mode="a",
                    header=write_header,
                    index=False
                )

        except Exception as e:
            errores.append({
                "idx_original": idx,
                "tickers": row.get("tickers_encontrados"),
                "error": str(e)
            })

    # Guardar errores
    if errores:
        file_err = Path(salida_err)
        write_header_err = not file_err.exists() or file_err.stat().st_size == 0
        pd.DataFrame(errores).to_csv(salida_err, mode="a", header=write_header_err, index=False)


    return pd.DataFrame(resultados)


In [18]:
df_final = procesar_news(df_news)
